# 11.9 Machine Learning for Time Series

**Machine Learning Mastery: Mathematics to MLOps** · Chapter 11: Time Series Machine Learning

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RithikRaj129/ml-mastery/blob/main/chapter-11-time-series-machine-learning/11.09_machine-learning-for-time-series.ipynb)

Companion code for Section 11.9 of the book. Run the cells in order from top to bottom.

**Example 11.9.1** · bash

```bash
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import xgboost as xgb
import lightgbm as lgb
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.model_selection import TimeSeriesSplit
import warnings; warnings.filterwarnings('ignore')

# ── Build full pipeline: load → features → split → train → eval ──
np.random.seed(42)
n = 730
dates = pd.date_range('2022-01-01', periods=n, freq='D')
t = np.arange(n)

# Realistic sales with multiple patterns
y = (1000 + 0.8*t
     + 300*np.sin(2*np.pi*t/365-np.pi/2)
     + 150*np.sin(2*np.pi*t/7)
     + np.random.normal(0, 60, n))

df = pd.DataFrame({'ds': dates, 'y': y}).set_index('ds')

# Add external regressors
df['temperature'] = 20 + 15*np.sin(2*np.pi*t/365) + np.random.normal(0, 3, n)
df['promotion']   = np.random.choice([0,0,0,0,1], n)  # 20% days have promotions
df.loc[df['promotion']==1, 'y'] *= 1.3  # Promotions boost sales by 30%

# ── Feature engineering ───────────────────────────────────────────
def make_features(df, target='y', lags=[1,7,14,28,90,365]):
    X = pd.DataFrame(index=df.index)
    # Calendar
    X['dow']     = df.index.dayofweek
    X['month']   = df.index.month
    X['doy']     = df.index.dayofyear
    X['dow_sin'] = np.sin(2*np.pi*X['dow']/7)
    X['dow_cos'] = np.cos(2*np.pi*X['dow']/7)
    X['doy_sin'] = np.sin(2*np.pi*X['doy']/365.25)
    X['doy_cos'] = np.cos(2*np.pi*X['doy']/365.25)
    X['trend']   = np.arange(len(df))
    # Lags
    for lag in lags:
        X[f'lag_{lag}'] = df[target].shift(lag)
    # Rolling
    for w in [7, 28]:
        X[f'roll_mean_{w}'] = df[target].shift(1).rolling(w).mean()
        X[f'roll_std_{w}']  = df[target].shift(1).rolling(w).std()
    # External regressors
    if 'temperature' in df.columns: X['temperature'] = df['temperature']
    if 'promotion'   in df.columns: X['promotion']   = df['promotion']
    return X

X = make_features(df)
y = df['y']
valid_idx = X.dropna().index
X, y = X.loc[valid_idx], y.loc[valid_idx]

# Walk-forward split
n_test = 90  # Last 90 days
X_tr = X.iloc[:-n_test];  y_tr = y.iloc[:-n_test]
X_te = X.iloc[-n_test:];  y_te = y.iloc[-n_test:]

# ── Compare ML models ────────────────────────────────────────────
models = {
    'Linear Regression':  LinearRegression(),
    'Ridge':              Ridge(alpha=1.0),
    'Random Forest':      RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1),
    'Gradient Boosting':  GradientBoostingRegressor(n_estimators=300, learning_rate=0.05, max_depth=4,
random_state=42),
    'XGBoost':            xgb.XGBRegressor(n_estimators=300, learning_rate=0.05, max_depth=4, random_state=42,
verbosity=0),
    'LightGBM':           lgb.LGBMRegressor(n_estimators=300, learning_rate=0.05, num_leaves=31, random_state=42,
verbose=-1),
}

print('=== ML Model Comparison (90-day walk-forward test) ===')
print(f'{'Model':<22} {'MAE':>8} {'MAPE%':>8} {'RMSE':>8}')
print('='*52)
results = {}
for name, model in models.items():
    model.fit(X_tr, y_tr)
    pred = model.predict(X_te)
    mae  = mean_absolute_error(y_te, pred)
    mape = np.mean(np.abs((y_te-pred)/y_te))*100
    rmse = np.sqrt(mean_squared_error(y_te, pred))
    results[name] = {'mae':mae,'mape':mape,'rmse':rmse,'pred':pred}
    print(f'{name:<22} {mae:>8.2f} {mape:>8.2f} {rmse:>8.2f}')

# ── Walk-forward cross-validation ────────────────────────────────
print('\n=== Walk-Forward Cross-Validation (XGBoost) ===')
tscv = TimeSeriesSplit(n_splits=5, test_size=30)
xgb_model = xgb.XGBRegressor(n_estimators=200, learning_rate=0.05,
                               max_depth=4, random_state=42, verbosity=0)

cv_maes = []
for fold, (tr_idx, te_idx) in enumerate(tscv.split(X)):
    X_tr_cv, X_te_cv = X.iloc[tr_idx], X.iloc[te_idx]
    y_tr_cv, y_te_cv = y.iloc[tr_idx], y.iloc[te_idx]
    xgb_model.fit(X_tr_cv, y_tr_cv)
    pred_cv = xgb_model.predict(X_te_cv)
    mae_cv  = mean_absolute_error(y_te_cv, pred_cv)
    mape_cv = np.mean(np.abs((y_te_cv-pred_cv)/y_te_cv))*100
    cv_maes.append(mae_cv)
    print(f'  Fold {fold+1}: train={len(tr_idx)} days, test={len(te_idx)} days, MAE={mae_cv:.2f},
MAPE={mape_cv:.2f}%')

print(f'  Mean CV MAE: {np.mean(cv_maes):.2f} ± {np.std(cv_maes):.2f}')

# ── Visualise best model forecast ────────────────────────────────
best_model_name = min(results, key=lambda k: results[k]['mape'])
best_pred = results[best_model_name]['pred']

fig, axes = plt.subplots(2, 1, figsize=(16, 10))
fig.suptitle(f'ML Time Series Forecasting - Best: {best_model_name}',
            fontsize=14, fontweight='bold')

# Full series + forecast
ax = axes[0]
ax.plot(y_tr.index[-90:], y_tr.values[-90:], 'b-', lw=2, alpha=0.8, label='Training (last 90d)')
ax.plot(y_te.index, y_te.values, 'k-', lw=2, label='Actual')
ax.plot(y_te.index, best_pred, 'r--', lw=2.5, label=f'{best_model_name}
(MAPE={results[best_model_name]["mape"]:.2f}%)')
ax.axvline(y_tr.index[-1], color='gray', ls=':', lw=2)
ax.set_ylabel('Sales'); ax.set_title('90-Day Forecast')
ax.legend(fontsize=10); ax.grid(alpha=0.3)

# Model comparison bar chart
ax = axes[1]
model_names = list(results.keys())
mapes = [results[m]['mape'] for m in model_names]
colors = ['#43A047' if m==best_model_name else '#1976D2' for m in model_names]
bars = ax.bar(model_names, mapes, color=colors, edgecolor='white')
for bar, mape in zip(bars, mapes):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.05,
           f'{mape:.2f}%', ha='center', fontsize=9, fontweight='bold')
ax.set_ylabel('MAPE (%)'); ax.set_title('Model Comparison')
ax.set_xticklabels(model_names, rotation=30, ha='right')
ax.grid(alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('ml_timeseries.png', dpi=150, bbox_inches='tight')
plt.show()
```